In [92]:
import os
from datetime import datetime
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchmetrics import Accuracy
from collect_hand import get_label_from_config

In [93]:
NUM_CLASSES = len(get_label_from_config('config.yaml'))
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [94]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        list_label = NUM_CLASSES
        self.linear_stack = nn.Sequential(
            nn.Linear(63, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(p=0.6)
        )
        self.output = nn.Linear(128, list_label)

    def forward(self, x):
        x = self.flatten(x)
        x = self.linear_stack(x)
        x = self.output(x)
        return x

    def get_pred_label(self, x):
        """return predicted class vector"""
        logits = self(x)
        out = nn.Softmax(dim=1)(logits)
        return torch.argmax(out, dim=1)

In [95]:
class EarlyStopper:
    def __init__(self, max_count, epsilon=0.001):
        self.prev = np.inf
        self.counter = 0
        self.epsilon = epsilon
        self.max_count = max_count

    def early_stop(self, curr):
        if curr < self.prev:            # loss was reduce
            self.counter = 0
            self.prev = curr  
        elif (curr > self.prev + self.epsilon):
            self.counter += 1
            if self.counter >= self.max_count:
                return True              # Early stopping happens
        return False

In [96]:
# customDataset -> DataLoader -> train/val/test
class CustomDataset(Dataset):
    def __init__(self, PATH):
        super(CustomDataset, self).__init__()
        self.data = pd.read_csv(PATH)
        self.y = torch.from_numpy(self.data.iloc[:, 0].to_numpy())
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        y_sample = self.y[index]
        X_sample = torch.from_numpy(self.data.iloc[index, 1:].to_numpy(dtype=np.float32))
        return X_sample, y_sample
    

train = CustomDataset('sign_data/train_split.csv')
val = CustomDataset('sign_data/val_split.csv')
test = CustomDataset('sign_data/test_split.csv')

print(train.__len__())
print(val.__len__())
print(test.__len__())

1722
1009
512


In [97]:
train_loader = DataLoader(train,
                          batch_size=40,
                          shuffle=True)

val_loader = DataLoader(val,
                        batch_size=40,
                        shuffle=True)

test_loader = DataLoader(test,
                         batch_size=20,
                         shuffle=False)

# model setup
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optmizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [98]:
def train(train_set, val_set, model, optimizer, loss_function, early_stopper, epochs=300):
    # for tracking best performance
    MODEL_PATH = './models'
    os.makedirs(MODEL_PATH, exist_ok=True)
    best_state = 1000000
    timestamp = datetime.now().strftime('%d-%m_%H-%M')
    
    for epoch in range(epochs):
        # training
        model.train(True)
        losses = []
        acc = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(device)
        for idx, (X_batch, y_batch) in enumerate(train_set):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_function(y_pred, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            acc.update(model.get_pred_label(X_batch), y_batch)
        loss_avg = sum(losses)/len(losses)

        # validation
        model.train(False)
        losses_val = []
        acc_val = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(device)
        for idx, (X_val, y_val) in enumerate(val_set):
            X_val, y_val = X_val.to(device), y_val.to(device)
            y_val_pred = model(X_val)
            loss_val = loss_function(y_val_pred, y_val)
            losses_val.append(loss_val.item())
            acc_val.update(model.get_pred_label(X_val), y_val)
        loss_val_avg = sum(losses_val)/len(losses_val)

        # logging
        print(f"Epoch {epoch}: ")
        print(f"Accuracy ---- train:{acc.compute().item()}, val:{acc_val.compute().item()}")
        print('LOSS --------- train {},  valid {}'.format(loss_avg, loss_val_avg))
        print('--------------------------------------------------\n')

        # tracking best performance trick
        if loss_val_avg < best_state:
            BEST_MODEL_PATH = f'{MODEL_PATH}/model_{timestamp}_{model.__class__.__name__}_best'
            best_state = loss_val_avg
            torch.save(model.state_dict(), BEST_MODEL_PATH)


        # early stopping
        if early_stopper.early_stop(loss_val_avg) is True:
            print(f"Early stopping at epoch {epoch}, minimum = {early_stopper.prev}")
            break

    
    # last state
    LAST_MODEL_PATH = f'./{MODEL_PATH}/model_{timestamp}_{model.__class__.__name__}_last'
    torch.save(model.state_dict(), LAST_MODEL_PATH)
    print(acc_val.compute())
    return model, BEST_MODEL_PATH    

In [99]:
# start training
early_stopper = EarlyStopper(max_count=30, epsilon=0.01)
model, best_path = train(train_loader, val_loader, model, optmizer, criterion, early_stopper)

Epoch 0: 
Accuracy ---- train:0.20325203239917755, val:0.21209117770195007
LOSS --------- train 1.6030558158050885,  valid 1.5896743490145757
--------------------------------------------------

Epoch 1: 
Accuracy ---- train:0.2926829159259796, val:0.3508424162864685
LOSS --------- train 1.569654174826362,  valid 1.5316049456596375
--------------------------------------------------

Epoch 2: 
Accuracy ---- train:0.4065040647983551, val:0.5619425177574158
LOSS --------- train 1.4936343431472778,  valid 1.4236187384678767
--------------------------------------------------

Epoch 3: 
Accuracy ---- train:0.5557491183280945, val:0.6828542947769165
LOSS --------- train 1.322327731685205,  valid 1.2210443661763117
--------------------------------------------------

Epoch 4: 
Accuracy ---- train:0.6771196126937866, val:0.7700693607330322
LOSS --------- train 1.0977310362187298,  valid 0.9455911540068113
--------------------------------------------------

Epoch 5: 
Accuracy ---- train:0.74854820

In [104]:
acc_test = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(device)

# load best state
network = MLP().to(device)
network.load_state_dict(torch.load(best_path, weights_only=False))
losses_test = []

network.eval()
for X_test, y_test in test_loader:
    X_test, y_test = X_test.to(device), y_test.to(device)
    y_test_pred = network(X_test)
    loss_test = criterion(y_test_pred, y_test)
    losses_test.append(loss_test.item())
    acc_test.update(network.get_pred_label(X_test), y_test)

# logging
print('HAND GESTURES CLASSIFICATION - HUYIGW04')
print(f'{"Accuracy of model:":<30}{acc_test.compute().item():>10.4f}')
print(f'{"Loss of model:":<30}{(sum(losses_test)/len(losses_test)):>10.4f}')
print('============================')


HAND GESTURES CLASSIFICATION - HUYIGW04
Accuracy of model:                0.9883
Loss of model:                    0.0460
